# `fx_viz` demo — the real module, not the §1.1 spike

This is the finished `Strategy.visualize()` (not `notebooks/viz_spike.ipynb`,
which only probed the raw `StreamChart` library APIs). Run the cells below in
order, then try the interactive checks in the last markdown cell.


In [ ]:
import sys
sys.path.insert(0, "..")

from lib.data import load_1s_data, resample
from lib.engine import Engine
from lib.strategies import SmaCrossoverStrategy

raw = load_1s_data("../data/EURUSD_1s_2024.csv")
engine = Engine(resample(raw, "5m"), raw)
print(f"{engine.signal_df.height:,} 5m bars loaded")


## Declare what to chart

`line_columns`/`marker_columns` are the only new things vs. the base
`SmaCrossoverStrategy` — everything else (SL/TP, signal logic) is unchanged.


In [ ]:
class VizStrategy(SmaCrossoverStrategy):
    line_columns = ["sma_fast", "sma_slow"]
    marker_columns = ["long_signal", "short_signal"]

strategy = VizStrategy(fast_n=20, slow_n=50, sl_pips=10, tp_pips=20)


## Cell A — once per session

Creates the chart and runs the backtest for the trade overlay
(`show_trades=True`). Only the last 500 bars load at first — panning near an
edge should pull in more (§5 lazy-loading).


In [ ]:
chart = strategy.visualize(engine, show_trades=True)
chart


## Cell B — rerun this whenever you change the strategy

This is the hot-reload check: edit `VizStrategy` above (e.g. change
`fast_n`/`slow_n`), rerun that cell, then rerun this one. The chart in Cell A's
output should update in place — same zoom/pan/timeframe, new data.


In [ ]:
strategy.visualize(engine, chart=chart, show_trades=True)
print("replotted")


---
## Things to actually try, by hand, in the chart above

1. **Pan left** (drag right) toward the edge of the loaded data. Candles
   should keep appearing as you approach the edge — that's §5 lazy-loading
   fetching another chunk from the full dataset, not something baked in
   upfront.
2. **Click a different timeframe button** (`1m 5m 15m 1h 4h` in the topbar).
   The candles AND both SMA lines should change to genuinely different
   values, not just the same shape stretched — that's §4's "recompute
   signals for the new timeframe," not just a relabel.
3. **Zoom/pan to some spot, then rerun Cell B.** The view should NOT jump
   back to the default — that's the hot-reload contract (§3).
4. **Look for the trade overlay**: green/red arrow markers at trade entries,
   and faint green/red background shading for the trade's open duration
   (`show_trades=True`'s projection, §8.4).
5. **Markers for the raw crossover signals** (`long_signal`/`short_signal`,
   small circles) should be visible along the SMA lines, separate from the
   trade-entry arrows.

If anything looks off (missing topbar, console errors, chart not updating on
rerun, etc.) — that's useful, tell me exactly what you saw.
